In [1]:
from datetime import datetime
from lightningrod.utils import config
from dotenv import load_dotenv
from lightningrod import (
    LightningRod,
    ContinuousAnswerType,
    GdeltSeedGenerator,
    ForwardLookingQuestionGenerator,
    QuestionPipeline,
    QuestionRenderer,
    WebSearchLabeler,
    NewsContextGenerator,
    continuous_example,
)

load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

In [2]:
instructions = """
Generate forward-looking forecasting questions with exactly one correct numeric answer.

Each question must ask about a future event or measurement that is unresolved at the time of asking and will resolve within 3 months to a finalized, publicly reported numeric value.

The outcome must be materially important, widely reported, and of real-world consequence (e.g., affecting economies, markets, elections, public policy, science, or competitive results).

High-value domains include economics, financial markets, elections, sports, science/environment, and public policy.

STRICTLY DO NOT include:
- Questions that cannot fully resolve within 3 months
"""

good_examples = [
    continuous_example(
        "What will be the United States Consumer Price Index (CPI-U) year-over-year inflation rate, expressed as a percentage, for the month of March 2026 as reported by the U.S. Bureau of Labor Statistics?"
    ),
    continuous_example(
        "What will be the official closing value of the S&P 500 index on the final trading day of April 2026?"
    ),
    continuous_example(
        "What will be the unemployment rate, expressed as a percentage, in the United States for April 2026 according to the U.S. Bureau of Labor Statistics?"
    ),
    continuous_example(
        "How many seats will the Liberal Party win in the Canadian federal election scheduled for April 28, 2026, based on final certified results?"
    ),
    continuous_example(
        "What will be the target federal funds rate upper bound, expressed as a percentage, immediately following the Federal Open Market Committee meeting ending May 6, 2026?"
    ),
    continuous_example(
        "What will be the closing settlement price of Brent crude oil futures, in U.S. dollars per barrel, on June 30, 2026 as reported by ICE?"
    ),
    continuous_example(
        "How many total goals will be scored in the UEFA Champions League final scheduled for May 30, 2026, based on the official match result?"
    ),
]

bad_examples = [
    continuous_example(
        "Will inflation rise next month?",
        comment="Binary yes/no framing; no numeric answer.",
    ),
    continuous_example(
        "What will the inflation rate be this year?",
        comment="Measurement period exceeds 3 months; not resolvable within the required window.",
    ),
    continuous_example(
        "How much will the economy grow soon?",
        comment="Vague timing, no country, no metric, no units.",
    ),
    continuous_example(
        "How many points will LeBron James score in his next NBA game?",
        comment="Event timing and occurrence are uncertain; not guaranteed to resolve to a finalized, reported value.",
    ),
    continuous_example(
        "What will be the unemployment rate in Europe next month?",
        comment="Ambiguous geography and aggregation; not a single authoritative value.",
    ),
    continuous_example(
        "What will be the average price of oil over the next quarter?",
        comment="Averaging period not yet complete at resolution; no single finalized value.",
    ),
    continuous_example(
        "What will be the U.S. inflation rate around the end of next quarter?",
        comment="Imprecise timing ('around'); not strictly defined or verifiable.",
    ),
]

In [9]:
answer_type = ContinuousAnswerType()

pipeline = QuestionPipeline(
    seed_generator=GdeltSeedGenerator(
        start_date=datetime(2024, 7, 2),
        end_date=datetime(2025, 11, 30),
        interval_duration_days=7,
        articles_per_interval=15,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        questions_per_seed=5,
        answer_type=answer_type
    ),
    context_generators=[NewsContextGenerator(num_articles=10)],
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.9,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

dataset = lr.transforms.run(pipeline, max_seeds=200)  # Increase to ~10000 for a real run

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $2.40                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃  In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons  ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ GdeltSeedGenerato… │ Complete             │   2 │  28 │        0 │      0 │ -                  │       3s │  │
│  │ ForwardLookingQue… │ Complete             │  28 │ 130 │        5 │      0 │ date_close not     │       2s │  │
│  │                    │                      │     │     │          │        │ after event_date   │          │  │
│  │                    │                      │     │     │          │        │ (5)                │          │  │
│  │ WebSearchLabelerT… │ Complete             │ 130 │ 121 │        9 │      0 │ Undetermined label │      13s │  │
│  │                    │                      │     │     │          │        │ (5), unknown (1),  │          │  │
│  │                    │                      │     │     │          │        │ Low confidence:    │          │  │
│  │                    │                      │     │     │          │        │ 0.80 < 0.9 (1),    │          │  │
│  │                    │                      │     │     │          │        │ Low confidence:    │          │  │
│  │                    │                      │     │     │          │        │ 0.85 < 0.9 (1),    │          │  │
│  │                    │                      │     │     │          │        │ Resolution date is │          │  │
│  │                    │                      │     │     │          │        │ before seed        │          │  │
│  │                    │                      │     │     │          │        │ creation date (1)  │          │  │
│  │ NewsContextGenera… │ Complete             │ 121 │ 118 │        3 │      0 │ <failed_attempts>  │   6m 14s │  │
│  │                    │                      │     │     │          │        │                    │          │  │
│  │                    │                      │     │     │          │        │ <generation        │          │  │
│  │                    │                      │     │     │          │        │ number="1">        │          │  │
│  │                    │                      │     │     │          │        │ <exception>        │          │  │
│  │                    │                      │     │     │          │        │     Connection     │          │  │
│  │                    │                      │     │     │          │        │ error.             │          │  │
│  │                    │                      │     │     │          │        │ </exception>       │          │  │
│  │                    │                      │     │     │          │        │ <completion>       │          │  │
│  │                    │                      │     │     │          │        │     None           │          │  │
│  │                    │                      │     │     │          │        │ </completion>      │          │  │
│  │                    │                      │     │     │          │        │ </generation>      │          │  │
│  │                    │                      │     │  